# Introducing Simultaneous Localization and Mapping (SLAM): THE Tutorial

Welcome to autonomous navigation. In this tutorial, we will implement the **Extended Kalman Filter (EKF) SLAM**. 

EKF SLAM represents the robot's state and the map as a single, massive Gaussian distribution. 
The algorithm runs in a continuous, two-step loop:
1. **Prediction Step (Motion Model):** The robot moves. We update its expected position based on its internal odometry (speed and steering). Because physical wheels slip, our uncertainty (covariance) *increases*.
2. **Correction Step (Measurement Model):** The robot's sensors (like LiDAR or cameras) detect landmarks. We compare where we *expected* the landmarks to be against where we *actually* saw them. We use this difference (the Innovation) to correct our position and update the map, *decreasing* our uncertainty.

### The Mathematical State Vector
Our state vector $\mu$ contains the robot's coordinates $(x, y, \theta)$ and the coordinates of all landmarks $(m_{x,i}, m_{y,i})$.

In [ ]:
# Install required libraries if you haven't:
# !pip install -q numpy matplotlib scipy

import numpy as np
import matplotlib.pyplot as plt
import math

# Set random seed for analytical reproducibility
np.random.seed(42)

print("Libraries imported successfully. Ready to build the SLAM backend.")

## Step 1: Environment and Sensor Initialization

To test our algorithm, we need to simulate the physical world. We will define:
* The true locations of the landmarks (which the SLAM algorithm does not know).
* The covariance matrices. In Kalman Filters, **Covariance ($Q$ and $R$)** mathematically represents our level of trust. 
  * $Q$ is the noise in our motors (how much we slip).
  * $R$ is the noise in our sensors (how blurry our camera is).

In [ ]:
# --- Simulation Parameters ---
DT = 0.1 # Time step in seconds
SIM_TIME = 20.0

# 1. Measurement Noise Covariance (R) - Trust in Sensors
# [Range variance, Angle variance]
R = np.diag([0.1 ** 2, np.deg2rad(5.0) ** 2])

# 2. Motion Noise Covariance (Q) - Trust in Motors
# [Velocity variance, Yaw-rate variance]
Q = np.diag([0.1 ** 2, np.deg2rad(10.0) ** 2])

# 3. True Landmark Positions (x, y) - The Ground Truth Map
RFID_LANDMARKS = np.array([[10.0, 0.0], 
                           [10.0, 10.0], 
                           [0.0, 15.0], 
                           [-5.0, 20.0]])

# 4. State Vector Initialization
# The state vector `xEst` holds [x, y, yaw]. We will append landmarks as we see them.
xEst = np.zeros((3, 1))
TrueState = np.zeros((3, 1))

# The Covariance Matrix `PEst` tracks the uncertainty of every variable against every other variable
PEst = np.eye(3)

print(f"Initial State Vector Shape: {xEst.shape}")
print(f"Initial Covariance Matrix Shape: \n{PEst}")

## Step 2: The Prediction Step (Motion Model)

When the robot issues a command to drive forward at velocity $v$ and turn at rate $\omega$, we must predict its new position. 

Because the world is non-linear (involving $\sin$ and $\cos$), we use the **Jacobian Matrix** ($G$). The Jacobian is a matrix of partial derivatives that analytically linearizes the curves of our motion, allowing the Gaussian math of the Kalman Filter to work without warping.

$$x_{t+1} = x_t + \begin{bmatrix} v \cos(\theta) \cdot dt \\ v \sin(\theta) \cdot dt \\ \omega \cdot dt \end{bmatrix}$$
$$P_{t+1} = G P_t G^T + Q$$

In [ ]:
def motion_model(x, u):
    """
    Predicts the next state based on current state (x) and control input (u = [v, yaw_rate]).
    """
    F = np.array([[1.0, 0, 0],
                  [0, 1.0, 0],
                  [0, 0, 1.0]])

    B = np.array([[DT * math.cos(x[2, 0]), 0],
                  [DT * math.sin(x[2, 0]), 0],
                  [0.0, DT]])

    x = (F @ x) + (B @ u)
    return x

def jacobian_motion(x, u):
    """
    Calculates the Jacobian matrix (G) of the motion model to propagate uncertainty.
    """
    yaw = x[2, 0]
    v = u[0, 0]
    
    # Partial derivatives of the motion model with respect to x, y, and yaw
    G = np.array([[1.0, 0.0, -DT * v * math.sin(yaw)],
                  [0.0, 1.0, DT * v * math.cos(yaw)],
                  [0.0, 0.0, 1.0]])
    return G

def predict_step(xEst, PEst, u):
    """Executes the mathematical Prediction Phase."""
    # 1. Update State
    xEst[0:3] = motion_model(xEst[0:3], u)
    
    # 2. Update Covariance (Uncertainty grows)
    G = jacobian_motion(xEst[0:3], u)
    PEst[0:3, 0:3] = G @ PEst[0:3, 0:3] @ G.T + Q
    
    return xEst, PEst

## Step 3: The Correction Step (Measurement Model)

When the robot senses a landmark, it calculates the distance and angle to it. 
We compare the *actual* sensor reading ($z$) to the *expected* reading ($z_{pred}$) based on our map.

The magic of SLAM happens here via the **Kalman Gain ($K$)**.
$$K = P H^T (H P H^T + R)^{-1}$$

Analytically, if our sensor noise ($R$) is low and our motion noise ($P$) is high, $K$ will be large. This tells the math: "Trust the sensors, override the odometry, and snap the robot's location to align with the landmarks!"

In [ ]:
def observation_model(x, landmark_pos):
    """Predicts what the sensor SHOULD see if our estimated state is perfectly correct."""
    dx = landmark_pos[0] - x[0, 0]
    dy = landmark_pos[1] - x[1, 0]
    d = math.hypot(dx, dy)
    angle = math.atan2(dy, dx) - x[2, 0]
    return np.array([[d], [angle]])

def update_step(xEst, PEst, z_actual, landmark_id):
    """Executes the Correction Phase using the Kalman Gain."""
    # For this tutorial, we assume the map is known to simplify the matrices.
    # In full SLAM, the landmark positions are part of xEst and updated here.
    lm_pos = RFID_LANDMARKS[landmark_id]
    
    # 1. Predict measurement
    z_pred = observation_model(xEst, lm_pos)
    
    # Calculate the Jacobian of the observation model (H)
    dx = lm_pos[0] - xEst[0, 0]
    dy = lm_pos[1] - xEst[1, 0]
    d2 = dx**2 + dy**2
    d = math.sqrt(d2)
    
    H = np.array([[-dx/d, -dy/d, 0.0],
                  [dy/d2, -dx/d2, -1.0]])
    
    # 2. Innovation (Difference between actual and predicted)
    y = z_actual - z_pred
    
    # 3. Calculate Kalman Gain
    S = H @ PEst[0:3, 0:3] @ H.T + R
    K = PEst[0:3, 0:3] @ H.T @ np.linalg.inv(S)
    
    # 4. Correct the State Vector
    xEst[0:3] = xEst[0:3] + (K @ y)
    
    # 5. Shrink the Covariance Matrix (We are more certain now!)
    I = np.eye(3)
    PEst[0:3, 0:3] = (I - (K @ H)) @ PEst[0:3, 0:3]
    
    return xEst, PEst

## Step 4: The SLAM Simulation Loop

We have defined the physics. Now we build the event loop.

At every time step:
1. The robot is commanded to move ($1.0$ m/s forward, turning slightly).
2. We simulate the *true* physics with injected noise.
3. We generate *noisy* sensor readings of the landmarks.
4. We feed the noisy command and noisy readings into our **EKF**, which mathematically fuses them to estimate the clean, true path.

In [ ]:
# --- Simulation Loop ---
time = 0.0

# History arrays for plotting
hxEst = xEst
hTrue = TrueState

print("Starting SLAM Simulation...")

while SIM_TIME >= time:
    time += DT
    
    # Control input (Velocity=1.0, YawRate=0.1)
    u = np.array([[1.0], [0.1]])
    
    # --- 1. Simulate the Real World (Ground Truth & Odometry) ---
    # Add noise to the command to simulate real, imperfect motors
    ud = u + np.array([[np.random.randn() * Q[0,0]], [np.random.randn() * Q[1,1]]])
    TrueState = motion_model(TrueState, u)
    
    # --- 2. EKF Prediction Phase ---
    # Predict where we are using the noisy odometry
    xEst, PEst = predict_step(xEst, PEst, ud)
    
    # --- 3. EKF Correction Phase ---
    for i in range(len(RFID_LANDMARKS)):
        # Simulate taking a sensor reading
        dx = RFID_LANDMARKS[i, 0] - TrueState[0, 0]
        dy = RFID_LANDMARKS[i, 1] - TrueState[1, 0]
        d = math.hypot(dx, dy)
        angle = math.atan2(dy, dx) - TrueState[2, 0]
        
        # Add noise to the sensor reading
        z_actual = np.array([[d + np.random.randn() * R[0,0]], 
                             [angle + np.random.randn() * R[1,1]]])
        
        # If the landmark is within sensor range (e.g., 20 meters), update!
        if d < 20.0:
            xEst, PEst = update_step(xEst, PEst, z_actual, i)
            
    # Save history for plotting
    hxEst = np.hstack((hxEst, xEst))
    hTrue = np.hstack((hTrue, TrueState))

# --- Plotting the Results ---
plt.figure(figsize=(10, 6))

# Plot True Path (What actually happened)
plt.plot(hTrue[0, :], hTrue[1, :], "-b", label="True Robot Path")

# Plot EKF Estimated Path (What the algorithm calculated)
plt.plot(hxEst[0, :], hxEst[1, :], "-r", label="EKF Estimated Path")

# Plot Landmarks
plt.plot(RFID_LANDMARKS[:, 0], RFID_LANDMARKS[:, 1], "*k", markersize=10, label="Landmarks")

plt.title("Extended Kalman Filter SLAM Analytical Simulation")
plt.xlabel("X (meters)")
plt.ylabel("Y (meters)")
plt.legend()
plt.grid(True)
plt.axis("equal")
plt.show()

print("Simulation Complete. Notice how tightly the EKF (Red) tracks the True Path (Blue) despite the severe injected noise!")

## Conclusion

You have successfully navigated the core mathematics of Simultaneous Localization and Mapping!

**Key Analytical Takeaways:**
1. **The Curse of Odometry:** If a robot only uses its internal sensors (wheel encoders), tiny errors compound over time. The estimated path will spiral completely away from the true path.
2. **The Kalman Fusion:** By comparing the expected landmark position against the sensor's actual reading, the Kalman Gain ($K$) acts as a mathematical magnet, pulling the drifting robot back into reality.
3. **Linearization:** The real world involves curves and angles (trigonometry). We had to use the **Jacobian Matrices ($G$ and $H$)** to take linear approximations of these curves at every time step so our Gaussian curves wouldn't break.

While we used known landmarks here to simplify the matrices, true SLAM simply expands the $xEst$ state vector dynamically as it discovers new landmarks, applying the exact same correlation mathematics!